# **SVR Regression Models**

**Import thư viện và thiết lập**

In [1]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.svm import SVR
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor

from sklearn.inspection import permutation_importance

# Config
SEED = 42
TARGET = "quantity_sold"
LEAK_COLS = ("review_to_sold_ratio",)
THRESHOLD = 2
TOP_K_LOW = 20
XGB_N_ITER = 10

TRAIN_CSV = "../data/processed/train_data_final.csv"
TEST_CSV  = "../data/processed/test_data_final.csv"

OUT_SVR_ENET = "./svr_enet_predictions.csv"
OUT_SVR_RF   = "./svr_rf_predictions.csv"
OUT_SVR_XGB  = "./svr_xgb_predictions.csv"

**Utility Functions**

In [2]:
def load_data(train_csv=TRAIN_CSV, test_csv=TEST_CSV, target=TARGET, leak_cols=LEAK_COLS):
    """Load và xử lý dữ liệu train/test"""
    train_df = pd.read_csv(train_csv)
    test_df  = pd.read_csv(test_csv)
    
    X_train = train_df.drop(columns=[target])
    y_train_log = train_df[target]
    X_test  = test_df.drop(columns=[target])
    y_test_log = test_df[target]
    
    for c in leak_cols:
        if c in X_train.columns: X_train = X_train.drop(columns=[c])
        if c in X_test.columns:  X_test  = X_test.drop(columns=[c])
    
    print(f"Data loaded: X_train{X_train.shape}, X_test{X_test.shape}")
    return X_train, y_train_log, X_test, y_test_log


def split_low_high(X_train, y_train_log, threshold=THRESHOLD):
    """Chia dữ liệu thành vùng Low và High"""
    mask_low  = y_train_log <= threshold
    mask_high = y_train_log > threshold
    
    X_low,  y_low_log  = X_train.loc[mask_low],  y_train_log.loc[mask_low]
    X_high, y_high_log = X_train.loc[mask_high], y_train_log.loc[mask_high]
    
    print(f"Low (<=2): {len(X_low)} | High (>2): {len(X_high)}")
    return X_low, y_low_log, X_high, y_high_log


def train_gate(X_train, y_train_log, threshold=THRESHOLD, seed=SEED):
    """Train Random Forest Classifier làm gate"""
    print("Training Gate...")
    y_low_binary = (y_train_log <= threshold).astype(int)
    gate = RandomForestClassifier(n_estimators=500, random_state=seed, n_jobs=-1)
    gate.fit(X_train, y_low_binary)
    print("✓ Gate trained")
    return gate


def save_predictions(y_test_log, y_pred_log, output_path):
    """Lưu predictions vào CSV"""
    pred_df = pd.DataFrame({
        "quantity_sold_ground_truth": y_test_log.values,
        "quantity_sold_predicted": y_pred_log
    })
    pred_df.to_csv(output_path, index=False)
    print(f"✓ Saved {len(pred_df)} predictions to: {output_path}")

**Functions for SVR Model**

In [3]:
def train_svr_low(X_low, y_low_log, top_k=TOP_K_LOW, seed=SEED):
    """Train SVR cho Low area"""
    print("\n" + "="*70)
    print("TRAINING LOW AREA: SVR")
    print("="*70)
    
    # Feature selection
    print("Step 1: Feature selection...")
    rf = RandomForestRegressor(n_estimators=400, random_state=seed, n_jobs=-1)
    rf.fit(X_low, y_low_log)
    fi = pd.Series(rf.feature_importances_, index=X_low.columns).sort_values(ascending=False)
    low_feats = fi.head(min(top_k, len(fi))).index.tolist()
    print(f"  Selected {len(low_feats)} features")
    
    # GridSearchCV
    print("Step 2: GridSearchCV...")
    pipe = Pipeline([("scaler", StandardScaler()), ("svr", SVR())])
    grid = {
        "svr__C": [0.1, 1, 10, 100],
        "svr__epsilon": [0.01, 0.1, 0.2],
        "svr__kernel": ["linear", "rbf", "poly"]
    }
    gs = GridSearchCV(pipe, grid, cv=3, scoring="neg_mean_squared_error", n_jobs=-1, verbose=1)
    gs.fit(X_low[low_feats], y_low_log)
    
    print(f"  Best params: {gs.best_params_}")
    print(f"  Best score: {-gs.best_score_:.6f}")
    
    # Permutation importance để refine features
    print("Step 3: Permutation importance for feature refinement...")
    perm = permutation_importance(gs.best_estimator_, X_low[low_feats], y_low_log, n_repeats=10, random_state=seed, scoring="neg_mean_squared_error")
    perm_imp = pd.Series(perm.importances_mean, index=low_feats).sort_values(ascending=False)
    
    # Refine features: chọn features có importance > threshold
    threshold = perm_imp.mean() * 0.1  # Loại bỏ features quá yếu
    refined_feats = perm_imp[perm_imp > threshold].index.tolist()
    
    if len(refined_feats) < 10:  # Đảm bảo tối thiểu 10 features
        refined_feats = perm_imp.head(min(20, len(perm_imp))).index.tolist()
    
    print(f"  Refined: {len(refined_feats)} features (from {len(low_feats)})")
    
    # In permutation importance (top 15)
    print("\n  Permutation importance (top 15):")
    perm_df = pd.DataFrame({"feature": perm_imp.index, "importance": perm_imp.values})
    print(perm_df.head(15).to_string(index=False))
    
    # Step 4: Re-train với refined features nếu có cải thiện
    if len(refined_feats) < len(low_feats):
        print("\nStep 4: Re-training with refined features...")
        gs_refined = GridSearchCV(pipe, grid, cv=3, scoring="neg_mean_squared_error", n_jobs=-1, verbose=0)
        gs_refined.fit(X_low[refined_feats], y_low_log)
        print(f"  Original score: {-gs.best_score_:.6f}")
        print(f"  Refined score:  {-gs_refined.best_score_:.6f}")
        
        if gs_refined.best_score_ > gs.best_score_:
            print("  ✓ Using refined model (better performance)")
            return gs_refined.best_estimator_, refined_feats
        else:
            print("  → Keeping original model")
    
    return gs.best_estimator_, low_feats

**Functions for linear regression model**

In [4]:
def train_elasticnet_high(X_high, y_high_log, seed=SEED):
    """Train ElasticNet cho High area"""
    print("\n" + "="*70)
    print("TRAINING HIGH AREA: ElasticNet")
    print("="*70)
    
    # ElasticNetCV
    print("Step 1: ElasticNetCV...")
    enet_cv = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNetCV(
            l1_ratio=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99],
            alphas=[0.0001, 0.001, 0.01, 0.1, 0.5, 1, 5, 10],
            cv=5, max_iter=10000, random_state=seed
        ))
    ])
    enet_cv.fit(X_high, y_high_log)
    best_alpha = enet_cv.named_steps["model"].alpha_
    best_l1 = enet_cv.named_steps["model"].l1_ratio_
    print(f"  Best: alpha={best_alpha}, l1_ratio={best_l1}")
    
    # Feature selection
    print("Step 2: Feature selection...")
    tmp = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=best_alpha, l1_ratio=best_l1, max_iter=10000, random_state=seed))
    ])
    tmp.fit(X_high, y_high_log)
    coef = tmp.named_steps["model"].coef_
    high_feats = X_high.columns[np.abs(coef) > 1e-8].tolist()
    if len(high_feats) == 0:
        high_feats = X_high.columns.tolist()
    print(f"  Selected {len(high_feats)} features")
    
    # Train final
    print("Step 3: Training final model with selected features...")
    high_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=best_alpha, l1_ratio=best_l1, max_iter=10000, random_state=seed))
    ])
    high_model.fit(X_high[high_feats], y_high_log)
    
    coef_df = pd.DataFrame({"feature": high_feats, "coef": high_model.named_steps["model"].coef_})
    coef_df["abs"] = coef_df["coef"].abs()
    coef_df = coef_df.sort_values("abs", ascending=False)
    
    # In coefficient importance (top 15)
    print("\n  Coefficient importance (top 15):")
    print(coef_df.head(15).to_string(index=False))
    
    # Step 4: Refine features dựa trên coefficient magnitude
    print("\nStep 4: Feature refinement based on coefficients...")
    coef_threshold_refined = coef_df["abs"].quantile(0.25)  # Loại bỏ 25% features yếu nhất
    refined_feats = coef_df[coef_df["abs"] >= coef_threshold_refined]["feature"].tolist()
    
    if len(refined_feats) < 10:
        refined_feats = coef_df.nlargest(15, "abs")["feature"].tolist()
    
    print(f"  Refined: {len(refined_feats)} features (from {len(high_feats)})")
    
    # Re-train với refined features
    if len(refined_feats) < len(high_feats):
        print("\nStep 5: Re-training with refined features...")
        refined_model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=best_alpha, l1_ratio=best_l1, max_iter=10000, random_state=seed))
        ])
        refined_model.fit(X_high[refined_feats], y_high_log)
        
        # So sánh performance (dùng train score vì không có validation set)
        orig_score = high_model.score(X_high[high_feats], y_high_log)
        refined_score = refined_model.score(X_high[refined_feats], y_high_log)
        
        print(f"  Original R²: {orig_score:.6f}")
        print(f"  Refined R²:  {refined_score:.6f}")
        
        if refined_score >= orig_score * 0.98:  # Cho phép giảm nhẹ nếu đơn giản hóa
            print("  ✓ Using refined model")
            return refined_model, refined_feats
        else:
            print("  → Keeping original model")
    
    return high_model, high_feats

**Functions for random forest regressor model**

In [5]:
def train_rf_high(X_high, y_high_log, seed=SEED):
    """Train Random Forest cho High area"""
    print("\n" + "="*70)
    print("TRAINING HIGH AREA: Random Forest")
    print("="*70)
    
    print("GridSearchCV for Random Forest...")
    rf = RandomForestRegressor(random_state=seed, n_jobs=-1)
    grid = {
        "n_estimators": [200, 500],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"]
    }
    gs = GridSearchCV(rf, grid, cv=3, scoring="r2", n_jobs=-1, verbose=1)
    gs.fit(X_high, y_high_log)
    
    high_feats = list(X_high.columns)
    fi = pd.Series(gs.best_estimator_.feature_importances_, index=high_feats).sort_values(ascending=False)
    
    print(f"  Best params: {gs.best_params_}")
    print(f"  Best score: {gs.best_score_:.6f}")
    
    # Feature refinement dựa trên importance
    print("\nFeature refinement...")
    importance_threshold = fi.quantile(0.20)  # Loại bỏ 20% features yếu nhất
    refined_feats = fi[fi >= importance_threshold].index.tolist()
    
    if len(refined_feats) < 15:
        refined_feats = fi.head(min(30, len(fi))).index.tolist()
    
    print(f"  Refined: {len(refined_feats)} features (from {len(high_feats)})")
    
    # In feature importance (top 15)
    print("\n  Feature importance (top 15):")
    fi_df = pd.DataFrame({"feature": fi.index, "importance": fi.values})
    print(fi_df.head(15).to_string(index=False))
    
    # Re-train với refined features nếu có cải thiện
    if len(refined_feats) < len(high_feats):
        print("\nRe-training with refined features...")
        gs_refined = GridSearchCV(rf, grid, cv=3, scoring="r2", n_jobs=-1, verbose=0)
        gs_refined.fit(X_high[refined_feats], y_high_log)
        
        print(f"  Original score: {gs.best_score_:.6f}")
        print(f"  Refined score:  {gs_refined.best_score_:.6f}")
        
        if gs_refined.best_score_ >= gs.best_score_ * 0.98:
            print("  ✓ Using refined model")
            return gs_refined.best_estimator_, refined_feats
        else:
            print("  → Keeping original model")
    
    return gs.best_estimator_, high_feats

**Functions for XGBoost regressor**

In [6]:
def train_xgb_high(X_high, y_high_log, n_iter=XGB_N_ITER, seed=SEED):
    """Train XGBoost cho High area"""
    print("\n" + "="*70)
    print("TRAINING HIGH AREA: XGBoost")
    print("="*70)
    
    # Initial training
    print("Step 1: Initial training...")
    X_tr, X_val, y_tr, y_val = train_test_split(X_high, y_high_log, test_size=0.2, random_state=seed)
    first_xgb = XGBRegressor(n_estimators=10000, learning_rate=0.1, max_depth=6, random_state=seed, early_stopping_rounds=50, n_jobs=-1)
    first_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    
    fi = pd.Series(first_xgb.feature_importances_, index=X_high.columns).sort_values(ascending=False)
    high_feats = fi.index.tolist()
    
    # In initial feature importance (top 15)
    print("\n  Initial feature importance (top 15):")
    fi_df = pd.DataFrame({"feature": fi.index, "importance": fi.values})
    print(fi_df.head(15).to_string(index=False))
    
    # RandomizedSearchCV
    print("Step 2: RandomizedSearchCV...")
    param_dist = {
        "n_estimators": [100, 500, 1000, 5000, 10000],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "max_depth": [3, 5, 7, 9],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0]
    }
    rs = RandomizedSearchCV(
        XGBRegressor(random_state=seed, n_jobs=-1),
        param_dist, n_iter=n_iter, scoring="neg_mean_squared_error",
        cv=3, random_state=seed, n_jobs=-1, verbose=1
    )
    rs.fit(X_tr[high_feats], y_tr)
    print(f"  Best params: {rs.best_params_}")
    
    # Train final
    print("\nStep 3: Training final model with all features...")
    high_model = XGBRegressor(**rs.best_params_, random_state=seed, n_jobs=-1)
    high_model.fit(X_high[high_feats], y_high_log)
    
    # Get final feature importance
    fi_final = pd.Series(high_model.feature_importances_, index=high_feats).sort_values(ascending=False)
    
    # Step 4: Refine features dựa trên importance
    print("\nStep 4: Feature refinement based on importance...")
    importance_threshold = fi_final.quantile(0.20)  # Loại bỏ 20% features yếu nhất
    refined_feats = fi_final[fi_final >= importance_threshold].index.tolist()
    
    if len(refined_feats) < 15:
        refined_feats = fi_final.head(min(25, len(fi_final))).index.tolist()
    
    print(f"  Refined: {len(refined_feats)} features (from {len(high_feats)})")
    
    # In feature importance (top 15)
    print("\n  Feature importance (top 15):")
    fi_df = pd.DataFrame({"feature": fi_final.index, "importance": fi_final.values})
    print(fi_df.head(15).to_string(index=False))
    
    # Re-train với refined features
    if len(refined_feats) < len(high_feats):
        print("\nStep 5: Re-training with refined features...")
        refined_model = XGBRegressor(**rs.best_params_, random_state=seed, n_jobs=-1)
        refined_model.fit(X_high[refined_feats], y_high_log)
        
        # So sánh performance
        orig_score = high_model.score(X_high[high_feats], y_high_log)
        refined_score = refined_model.score(X_high[refined_feats], y_high_log)
        
        print(f"  Original R²: {orig_score:.6f}")
        print(f"  Refined R²:  {refined_score:.6f}")
        
        if refined_score >= orig_score * 0.98:  # Cho phép giảm nhẹ để đơn giản hóa
            print("  ✓ Using refined model")
            return refined_model, refined_feats
        else:
            print("  → Keeping original model")
    
    return high_model, high_feats

**Functions for prediction**

In [7]:
def hybrid_predict(gate, low_model, high_model, X_test, low_feats, high_feats):
    """Dự đoán hybrid sử dụng gate"""
    print("\n" + "="*70)
    print("HYBRID PREDICTION")
    print("="*70)
    
    p_low = gate.predict_proba(X_test)[:, 1]
    y_low_pred_log  = low_model.predict(X_test[low_feats])
    y_high_pred_log = high_model.predict(X_test[high_feats])
    
    y_low_pred_real  = np.clip(np.expm1(y_low_pred_log), 0, None)
    y_high_pred_real = np.clip(np.expm1(y_high_pred_log), 0, None)
    y_pred_real = p_low * y_low_pred_real + (1 - p_low) * y_high_pred_real
    y_pred_log  = np.log1p(y_pred_real)
    
    print(f"Mean P(Low): {p_low.mean():.4f}, Predictions: {len(y_pred_log)}")
    return y_pred_log

## **1. SVR + ElasticNet**

In [8]:
# Load data
X_train, y_train_log, X_test, y_test_log = load_data()
X_low, y_low_log, X_high, y_high_log = split_low_high(X_train, y_train_log)

# Train models
gate = train_gate(X_train, y_train_log)
low_model, low_feats = train_svr_low(X_low, y_low_log)
high_model, high_feats = train_elasticnet_high(X_high, y_high_log)

# Predict & Save
y_pred_log = hybrid_predict(gate, low_model, high_model, X_test, low_feats, high_feats)
save_predictions(y_test_log, y_pred_log, OUT_SVR_ENET)

Data loaded: X_train(15848, 53), X_test(3963, 53)
Low (<=2): 8480 | High (>2): 7368
Training Gate...
✓ Gate trained

TRAINING LOW AREA: SVR
Step 1: Feature selection...
  Selected 20 features
Step 2: GridSearchCV...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
  Best params: {'svr__C': 1, 'svr__epsilon': 0.2, 'svr__kernel': 'rbf'}
  Best score: 0.258363
Step 3: Permutation importance for feature refinement...
  Refined: 20 features (from 20)

  Permutation importance (top 15):
                               feature  importance
                          review_count    0.095518
 category_root_name_Nhà Cửa - Đời Sống    0.028209
 category_root_name_Làm Đẹp - Sức Khỏe    0.023490
                    store_review_count    0.022712
                         total_visuals    0.020778
                      reputation_score    0.018483
                           image_count    0.017495
                     price_vs_category    0.017313
category_root_name_Thể Thao – Dã Ngoại    

c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 9.615132965661815, tolerance: 1.8935101460719435
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 7.38919939392872, tolerance: 1.9101929048782997
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 7.769364890039014, tolerance: 1.8930170870701948
  model = cd_fast.enet_coordinate_descent_gram(
c:

  Best: alpha=0.0001, l1_ratio=0.3
Step 2: Feature selection...


c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.297e+00, tolerance: 2.365e+00
  model = cd_fast.enet_coordinate_descent(


  Selected 50 features
Step 3: Training final model with selected features...


c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.297e+00, tolerance: 2.365e+00
  model = cd_fast.enet_coordinate_descent(



  Coefficient importance (top 15):
                                  feature      coef      abs
                             review_count  1.480468 1.480468
                        price_vs_category -0.814674 0.814674
                                    price  0.421678 0.421678
                       store_review_count  0.256614 0.256614
         category_root_name_Nhà Sách Tiki  0.247664 0.247664
   category_root_name_Điện Tử - Điện Lạnh -0.235255 0.235255
    category_root_name_Làm Đẹp - Sức Khỏe  0.213976 0.213976
                           original_price  0.202689 0.202689
      category_root_name_Chăm sóc nhà cửa  0.200929 0.200929
       category_root_name_Bách Hóa Online  0.194876 0.194876
category_root_name_Ô Tô – Xe Máy – Xe Đạp -0.188712 0.188712
    category_root_name_Nhà Cửa - Đời Sống  0.174954 0.174954
                           rating_average -0.156830 0.156830
                  category_root_name_NGON  0.131222 0.131222
     category_root_name_Đồ chơi - Mẹ & Bé  0.1239

## **2. SVR + Random Forest**

In [9]:
# Load data
X_train, y_train_log, X_test, y_test_log = load_data()
X_low, y_low_log, X_high, y_high_log = split_low_high(X_train, y_train_log)

# Train models
gate = train_gate(X_train, y_train_log)
low_model, low_feats = train_svr_low(X_low, y_low_log)
high_model, high_feats = train_rf_high(X_high, y_high_log)

# Predict & Save
y_pred_log = hybrid_predict(gate, low_model, high_model, X_test, low_feats, high_feats)
save_predictions(y_test_log, y_pred_log, OUT_SVR_RF)

Data loaded: X_train(15848, 53), X_test(3963, 53)
Low (<=2): 8480 | High (>2): 7368
Training Gate...
✓ Gate trained

TRAINING LOW AREA: SVR
Step 1: Feature selection...
  Selected 20 features
Step 2: GridSearchCV...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
  Best params: {'svr__C': 1, 'svr__epsilon': 0.2, 'svr__kernel': 'rbf'}
  Best score: 0.258363
Step 3: Permutation importance for feature refinement...
  Refined: 20 features (from 20)

  Permutation importance (top 15):
                               feature  importance
                          review_count    0.095518
 category_root_name_Nhà Cửa - Đời Sống    0.028209
 category_root_name_Làm Đẹp - Sức Khỏe    0.023490
                    store_review_count    0.022712
                         total_visuals    0.020778
                      reputation_score    0.018483
                           image_count    0.017495
                     price_vs_category    0.017313
category_root_name_Thể Thao – Dã Ngoại    

## **3. SVR + XGBoost**

In [10]:
# Load data
X_train, y_train_log, X_test, y_test_log = load_data()
X_low, y_low_log, X_high, y_high_log = split_low_high(X_train, y_train_log)

# Train models
gate = train_gate(X_train, y_train_log)
low_model, low_feats = train_svr_low(X_low, y_low_log)
high_model, high_feats = train_xgb_high(X_high, y_high_log)

# Predict & Save
y_pred_log = hybrid_predict(gate, low_model, high_model, X_test, low_feats, high_feats)
save_predictions(y_test_log, y_pred_log, OUT_SVR_XGB)

Data loaded: X_train(15848, 53), X_test(3963, 53)
Low (<=2): 8480 | High (>2): 7368
Training Gate...
✓ Gate trained

TRAINING LOW AREA: SVR
Step 1: Feature selection...
  Selected 20 features
Step 2: GridSearchCV...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
  Best params: {'svr__C': 1, 'svr__epsilon': 0.2, 'svr__kernel': 'rbf'}
  Best score: 0.258363
Step 3: Permutation importance for feature refinement...
  Refined: 20 features (from 20)

  Permutation importance (top 15):
                               feature  importance
                          review_count    0.095518
 category_root_name_Nhà Cửa - Đời Sống    0.028209
 category_root_name_Làm Đẹp - Sức Khỏe    0.023490
                    store_review_count    0.022712
                         total_visuals    0.020778
                      reputation_score    0.018483
                           image_count    0.017495
                     price_vs_category    0.017313
category_root_name_Thể Thao – Dã Ngoại    